<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 2 · EXPLORING AND VISUALISING DATA</span>

# Session 4: Choose the right chart and tell the story

**Dataset:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). You work with a 10,000-row seeded sample, `ev_m2_sample.csv`.
**Time:** 60 minutes. **Course outcomes:** CO2 (and CO3 for Session 5).

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Berkeley: the wrong grouping tells the wrong story |
| Concept | 10 | Which chart for which question (instructor) |
| Guided lab | 30 | Build four charts, write a headline for each |
| Team task | 10 | Chart-off: fix a misleading chart |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- match a question to a chart type
- build four charts and give each a one-sentence headline
- spot a misleading axis and fix it
- explain why a group difference can disappear when you split the data

**New commands this session (max 5):** grouped bar, `sns.lineplot`, `pivot_table`, `sns.heatmap`, titles and labels

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [0]:
pip install seaborn


In [0]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps


FILE = "/Volumes/workspace/ev_sales/ev_sales_data/ev_m2_sample.csv"
raw = pd.read_csv(FILE)
df = raw.copy()
print("Loaded", df.shape[0], "rows x", df.shape[1], "columns")

## Part A. Setup

In [0]:
def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

In [0]:
def apply_s3_decisions(d):
    """Session 3 data-quality decisions, applied in one place (see your decision log)."""
    d = d.copy()
    d["income_was_missing"] = d["Annual_Income_USD"].isna()
    d["Annual_Income_USD"] = d["Annual_Income_USD"].fillna(d.groupby("City_Type")["Annual_Income_USD"].transform("median"))
    d["Daily_Commute_km"] = d["Daily_Commute_km"].fillna(d["Daily_Commute_km"].median())
    d = d.dropna(subset=["Environmental_Concern_Level"])      # strongest signal: do not blur it with a median
    d["income_at_floor"] = d["Annual_Income_USD"] == 30000
    d["commute_at_floor"] = d["Daily_Commute_km"] == 5
    return d

df = add_derived(apply_s3_decisions(raw))     # the Session 3 decisions, pre-applied
df["Subsidy"] = df["Subsidy_Available"].map({"Yes": "Subsidy", "No": "No subsidy"})
print(df.shape)

## The chart rule (read, 3 min)
| Your question | Chart |
|---|---|
| How does a rate compare across categories? | **bar** (start the axis at 0) |
| How does a rate change across an ordered scale? | **line** |
| How are values spread? | **histogram / box** |
| How do two numbers move together? | **scatter** |
| How does a rate depend on two categories? | **heatmap** |

Every chart gets: a **question**, the **right chart**, one **headline** that says the finding.

## Chart 1. Which city type buys most?

In [0]:
rate = df.groupby("City_Type")["buy"].mean().reindex(["Rural", "Suburban", "Urban"])
n = df["City_Type"].value_counts()
fig, ax = plt.subplots(figsize=(6, 3.6))
sns.barplot(x=rate.index, y=rate.values, color=C["terracotta"], ax=ax)
for i, c in enumerate(rate.index):
    ax.text(i, rate[c] + 0.004, f"{rate[c]:.1%}", ha="center", fontsize=10)
ax.set_ylim(0, ____)       # TODO: bars must start at 0. What top value leaves room for the labels?
ax.set_ylabel("Share saying Yes"); ax.yaxis.set_major_formatter(PCT); ax.set_xlabel("")
ax.set_title("____")       # TODO: write a headline that states the finding
plt.tight_layout()

## Chart 2. Is it the city, or something else? (the star chart)
**Predict first:** if you split each city by *home charging possible*, will the city gaps stay?

In [0]:
g = df.groupby(["City_Type", "Home_Charging_Possible"])["buy"].agg(rate="mean", n="size").reset_index()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
sns.barplot(data=g, x="City_Type", y="rate", hue="Home_Charging_Possible", order=["Rural", "Suburban", "Urban"],
            hue_order=["Yes", "No"], palette={"Yes": C["lagoon"], "No": C["ochre"]}, ax=ax)
for p, (_, row) in zip(ax.patches, g.sort_values(["Home_Charging_Possible", "City_Type"], key=lambda s: s.map({"Yes": 0, "No": 1, "Rural": 0, "Suburban": 1, "Urban": 2})).iterrows()):
    ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.004, f"{p.get_height():.0%}\nn={int(row['n']):,}", ha="center", fontsize=8)
ax.set_ylim(0, 0.30); ax.set_ylabel("Share saying Yes"); ax.yaxis.set_major_formatter(PCT); ax.set_xlabel("")
ax.legend(title="Can charge at home?", frameon=False)
ax.set_title("____")       # TODO: write the headline this chart earns
plt.tight_layout()

In [0]:
# How common is home charging in each city type?
(df["Home_Charging_Possible"] == "Yes").groupby(df["City_Type"]).mean().round(2)

**YOUR ANSWER:** Chart 1 said 'rural people prefer EVs'. What does Chart 2 say instead? Write the corrected headline for the marketing head.

> _write here_

## Chart 3. How much do subsidy and environmental concern matter?

In [0]:
rate = df.groupby(["Environmental_Concern_Level", "Subsidy"])["buy"].mean().reset_index()
fig, ax = plt.subplots(figsize=(7, 3.8))
sns.lineplot(data=rate, x="Environmental_Concern_Level", y="buy", hue="Subsidy", style="Subsidy", markers=True, dashes=False,
             palette={"Subsidy": C["terracotta"], "No subsidy": C["lagoon"]}, ax=ax, legend=False)
ax.text(5.05, rate[(rate.Subsidy == "Subsidy") & (rate.Environmental_Concern_Level == 5)]["buy"].iloc[0], "Subsidy available", color=C["ink"], va="center", fontsize=9)
ax.text(5.05, rate[(rate.Subsidy == "No subsidy") & (rate.Environmental_Concern_Level == 5)]["buy"].iloc[0], "No subsidy", color=C["ink"], va="center", fontsize=9)
ax.set_xlim(0.8, 6.2); ax.set_xticks([1, 2, 3, 4, 5]); ax.set_ylim(0, 0.9)
ax.set_xlabel("Environmental concern (1 low to 5 high)"); ax.set_ylabel("Share saying Yes"); ax.yaxis.set_major_formatter(PCT)
ax.set_title("Concern only turns into a Yes when a subsidy is available")
plt.tight_layout()

## Chart 4. Income and concern together (heatmap)

In [0]:
pt = df.pivot_table(values="buy", index="Environmental_Concern_Level", columns="income_band", aggfunc="____", observed=True)    # TODO: which summary gives the Yes rate?
pt = pt.sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(6.6, 4))
sns.heatmap(pt, annot=True, fmt=".0%", cmap=SEQ, cbar_kws={"label": "Share saying Yes"}, linewidths=2, linecolor=C["sand"], ax=ax)
ax.set_xlabel("Income band (quartiles)"); ax.set_ylabel("Environmental concern")
ax.set_title("____")       # TODO: headline
plt.tight_layout()

## Chart-off (10 min)
Below is a chart a junior analyst sent to the marketing head. **Find three problems, then redraw it.**

In [0]:
def weak_chart():
    r = df.groupby("City_Type")["buy"].mean()
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.bar(r.index, r.values, color=["red", "green", "blue"])
    ax.set_ylim(0.15, 0.22)
    ax.set_title("EV")
    plt.tight_layout()
weak_chart()

**Problems you found (list three):**

> 1. 
> 2. 
> 3. 

In [0]:
def better_chart():
    r = df.groupby("City_Type")["buy"].mean().reindex(["Rural", "Suburban", "Urban"])
    fig, ax = plt.subplots(figsize=(6, 3.6))
    # TODO: draw a clearer version: one colour, bars from zero, value labels, axis label, a headline title
    plt.tight_layout()
better_chart()

**Score your team's chart (0-2 each):** clear question __ · right chart __ · clear takeaway __ · **total /6**

## Output: your four-chart mini-story
For each chart, one sentence a marketing head can use. Then save the charts.

In [0]:
story = pd.DataFrame({
    "chart": ["1 City", "2 City x home charging", "3 Concern x subsidy", "4 Income x concern"],
    "headline": ["", "", "", ""],      # TODO: fill four headlines
})
story

## Exit ticket (5 min)
1. Which chart would you use to compare Yes rates across five income bands, and why not a pie?
2. Chart 1 and Chart 2 gave different headlines. Which one should the marketing head see, and what does Chart 1 do wrong?
3. Why print n on a bar?